# Coingecko API Notebook

This notebook demonstrates how to retrieve real-time cryptocurrency data using the [Coingecko API](https://www.coingecko.com/en/api).

We will:
- Fetch live Bitcoin price data.
- Parse and visualize trends over time.
- Optionally stream or store the data.


This notebook follows best practices outlined in [causify-ai/helpers](https://github.com/causify-ai/helpers/blob/master/docs/coding/all.jupyter_notebook.how_to_guide.md).

> Citation: Coingecko API. [https://www.coingecko.com/en/api](https://www.coingecko.com/en/api)

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

## Imports

In [25]:
import logging
# Import libraries in this section.
# Avoid imports like import *, from ... import ..., from ... import *, etc.

import requests
import time
from datetime import datetime
import pandas as pd

## Load Bitcoin data every minute 

In [61]:
import os

API_KEY = os.getenv("Coingecko_API_KEY ")
BASE_URL = "https://api.coingecko.com/api/v3"

In [42]:
def get_bitcoin_price(vs_currency: str = "usd") -> float:
    """
    Fetches the current Bitcoin price in the given currency from CoinGecko.
    """
    endpoint = f"{BASE_URL}/simple/price"
    params = {"ids": "bitcoin", "vs_currencies": vs_currency}
    resp = requests.get(endpoint, params=params)
    resp.raise_for_status()
    return resp.json()["bitcoin"][vs_currency]


In [43]:
poll_interval = 60

try:
    while True:
        # 1. Current price
        price = get_bitcoin_price()
        print(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] BTC price: ${price:,.2f}")

        time.sleep(poll_interval)

except KeyboardInterrupt:
    print("Monitoring stopped by user.")


[2025-04-30 08:00:13] BTC price: $94,609.00
[2025-04-30 08:01:14] BTC price: $94,611.00
[2025-04-30 08:02:14] BTC price: $94,615.00
Monitoring stopped by user.


## Get Coin’s OHLC Data with an API

In [33]:
def get_coin_ohlc(coin_id: str = "bitcoin",
                  vs_currency: str = "usd",
                  days: int = 30) -> list:
    """
    Fetches OHLC data for the specified coin over the last `days` days.
    Returns a list of [timestamp, open, high, low, close].
    """
    endpoint = f"{BASE_URL}/coins/{coin_id}/ohlc"
    params = {"vs_currency": vs_currency, "days": days}
    resp = requests.get(endpoint, params=params)
    resp.raise_for_status()
    return resp.json()

In [37]:
# 2. OHLC for last 7 days
ohlc = get_coin_ohlc(days=7)
print(f"  • Retrieved {len(ohlc)} OHLC data points for past 7 days")
df = pd.DataFrame(ohlc)
df.columns = [ "date", "open", "high", "low", "close"]
df["date"] = pd.to_datetime(df["date"], unit = "ms")
df.set_index('date', inplace = True)

# Display the dataframe with better formatting
print("\nBitcoin OHLC Data (7 days):")
display(df)

# You can also create a quick summary
print("\nSummary Statistics:")
display(df.describe())

  • Retrieved 42 OHLC data points for past 7 days

Bitcoin OHLC Data (7 days):


,open,high,low,close
date,,,,
2025-04-23 08:00:00,93161.0,93818.0,93103.0,93818.0
2025-04-23 12:00:00,93952.0,94320.0,93454.0,93454.0
2025-04-23 16:00:00,93491.0,94020.0,92079.0,92754.0
2025-04-23 20:00:00,93025.0,94122.0,93025.0,93512.0
2025-04-24 00:00:00,93534.0,93869.0,93379.0,93605.0
...,...,...,...,...
2025-04-29 12:00:00,94961.0,95228.0,94819.0,95119.0
2025-04-29 16:00:00,95139.0,95158.0,94737.0,94924.0
2025-04-29 20:00:00,94884.0,95444.0,94882.0,95352.0



Summary Statistics:


,open,high,low,close
count,42.0,42.00,42.00,42.00
mean,94152.1,94524.79,93753.33,94178.81
std,768.5,734.94,782.11,780.55
min,92216.0,92612.00,91810.00,92194.00
25%,93711.5,94042.50,93148.25,93723.00
50%,94199.5,94464.50,93880.00,94247.00
75%,94723.0,95153.75,94344.50,94796.25
max,95345.0,95564.00,94882.00,95352.00


# Get Historical Bicoin Data

In [65]:
def get_historical_data(
    coin_id: str = "bitcoin",
    date_str: str = None
) -> dict:
    """
    Fetches historical data for `coin_id` on a given date (DD-MM-YYYY).
    If no date_str is provided, defaults to today.
    Returns the full JSON payload, including market_data.
    """
    if date_str is None:
        date_str = datetime.utcnow().strftime("%d-%m-%Y")

    endpoint = f"{BASE_URL}/coins/{coin_id}/history"
    params = {"date": date_str}

    headers = {}
    if API_KEY:
        # CoinGecko Pro expects this exact header name
        headers["X-Cg-Pro-Api-Key"] = API_KEY

    resp = requests.get(endpoint, params=params, headers=headers)
    resp.raise_for_status()
    return resp.json()

In [66]:
# 3. Historical snapshot (e.g., today’s date last year)
#    Example: use date_str="01-01-2025" or leave None for today
hist = get_historical_data(date_str="01-01-2025")
hist_price = hist.get("market_data", {}).get("current_price", {}).get("usd")
print(f"  • BTC price on 01-01-2025: ${hist_price:,.2f}" if hist_price else "  • No data for 01-01-2025")

  • BTC price on 01-01-2025: $93,507.86


## Fetch Cryptocurrency Bitcoin Price Data 

In [41]:
def get_market_chart(coin_id: str = "bitcoin",
                     vs_currency: str = "usd",
                     days: int = 30) -> dict:
    """
    Fetches market chart data (prices, market caps, total volumes)
    for the past `days` days.
    Returns a dict with keys: 'prices', 'market_caps', 'total_volumes'.
    """
    endpoint = f"{BASE_URL}/coins/{coin_id}/market_chart"
    params = {"vs_currency": vs_currency, "days": days}
    resp = requests.get(endpoint, params=params)
    resp.raise_for_status()
    return resp.json()

In [56]:
# 4. Market chart for past 30 days
chart = get_market_chart(days=30)
print(f"  • Market chart contains {len(chart['prices'])} price points over 30 days")

# Create a DataFrame from the chart data
# The chart data has 'prices', 'market_caps', and 'total_volumes' as keys
# Each value is a list of [timestamp, value] pairs
prices_data = [[item[0], item[1]] for item in chart['prices']]
market_caps_data = [[item[0], item[1]] for item in chart['market_caps']]
total_volumes_data = [[item[0], item[1]] for item in chart['total_volumes']]

# Create DataFrames for each metric
prices_df = pd.DataFrame(prices_data, columns=['date', 'price'])
market_caps_df = pd.DataFrame(market_caps_data, columns=['date', 'market_cap'])
total_volumes_df = pd.DataFrame(total_volumes_data, columns=['date', 'volume'])

# Convert timestamp to datetime
prices_df['date'] = pd.to_datetime(prices_df['date'], unit='ms')
market_caps_df['date'] = pd.to_datetime(market_caps_df['date'], unit='ms')
total_volumes_df['date'] = pd.to_datetime(total_volumes_df['date'], unit='ms')

# Merge the DataFrames
market_df = pd.merge(prices_df, market_caps_df, on='date')
market_df = pd.merge(market_df, total_volumes_df, on='date')

# Set date as index
market_df.set_index('date', inplace=True)

# Display latest data with date
latest_date = market_df.index[-1].strftime('%Y-%m-%d %H:%M:%S')
print(f"\nLatest Bitcoin Data (as of {latest_date}):")
print(f"Price: ${market_df['price'].iloc[-1]:,.2f}")
print(f"Market Cap: ${market_df['market_cap'].iloc[-1]:,.2f}")
print(f"24h Volume: ${market_df['volume'].iloc[-1]:,.2f}")

# Display the dataframe
market_df

  • Market chart contains 721 price points over 30 days

Latest Bitcoin Data (as of 2025-04-30 08:20:54):
Price: $94,712.33
Market Cap: $1,880,798,816,292.57
24h Volume: $23,650,636,999.45


,price,market_cap,volume
date,,,
2025-03-31 09:04:49.639,81882.35,1.63e+12,1.99e+10
2025-03-31 10:04:21.646,81488.96,1.61e+12,2.06e+10
2025-03-31 11:04:52.578,82184.65,1.63e+12,1.96e+10
2025-03-31 12:04:26.791,82227.54,1.63e+12,2.17e+10
2025-03-31 13:05:43.149,82741.55,1.64e+12,2.06e+10
...,...,...,...
2025-04-30 05:04:36.956,95035.54,1.89e+12,2.47e+10
2025-04-30 06:06:29.945,94946.47,1.89e+12,2.35e+10
2025-04-30 07:19:28.119,94835.46,1.88e+12,2.34e+10


## The flow should be highlighted using headings in markdown
```
# Level 1
## Level 2
### Level 3
```